In [1]:
# =============================================================================
# QUANTIZATION & INFERENCE OPTIMIZATION — make Correction-GPT faster/smaller
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHERE ARE WE ON THE PATH?
# ---------------------------------------------------------------------------
#   Pretrain → SFT → DPO  = teach the model WHAT to say
#   THIS NOTEBOOK         = serve it CHEAPER / FASTER on a device
#
# Easy analogy:
#   Training notebooks = write the book.
#   Quantization       = print a pocket edition (smaller file, still readable).
#   Inference opt      = read it faster (less waiting for the next word).
#
# Why care for your earpiece product?
#   Live "Correction: ..." whispers need LOW latency on phone/edge hardware.
#   Full float32 weights are accurate but heavy. Quantization shrinks them.
#
#
# ---------------------------------------------------------------------------
# THIS CELL — load the trained model + measure BASELINE speed/size
# ---------------------------------------------------------------------------
# Before optimizing, measure the "before" numbers:
#   1) How long to generate ~15 tokens?  (latency in ms)
#   2) How many parameters?
#   3) How many MB if each weight is float32 (4 bytes)?
# Later cells compare quantized / optimized versions to THIS baseline.
#

from pathlib import Path
import sys
import time

import torch

# week2/ on path — same pattern as notebooks 9–10
# (You cannot `import` a .ipynb; shared code lives in .py modules.)
_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

# MiniGPT = model class from notebook 7, saved as week2/mini_gpt.py
#   NOT day8_minigpt — that module name does not exist.
from mini_gpt import MiniGPT

# BPETokenizer = from notebook 8, saved as week2/tokenizer.py
#   Sticky lesson from notebook 8: keep </w> inside merges (don't strip)
#   or decode glues words like "theapirate...".
import importlib
import tokenizer as _tokenizer_mod
importlib.reload(_tokenizer_mod)  # pick up save/load if kernel had an old copy
from tokenizer import BPETokenizer

# ---------------------------------------------------------------------------
# Load the SAME tokenizer the DPO/SFT weights were trained with
# ---------------------------------------------------------------------------
# Do NOT train on "placeholder" text — that makes a NEW vocab (wrong size)
# and torch.load will crash with shape mismatch (260 vs something else).
#
# Picture:
#   correction_gpt_sft_tokenizer.json  →  jersey numbers (token → id)
#   correction_gpt_dpo.pt              →  weights that EXPECT those ids
#

_tok_candidates = [
    Path("correction_gpt_sft_tokenizer.json"),
    Path("week2") / "correction_gpt_sft_tokenizer.json",
]
_tok_path = next((p for p in _tok_candidates if p.is_file()), None)
if _tok_path is None:
    raise FileNotFoundError(
        "Need correction_gpt_sft_tokenizer.json (from notebook 9). "
        "Run SFT save cell or keep the file in week2/."
    )

tokenizer = BPETokenizer()
tokenizer.load(_tok_path)

vocab_size = len(tokenizer.vocab)
block_size = 64  # must match checkpoint pos_embed length

model = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)

# Prefer DPO weights; fall back to SFT if DPO file missing
_ckpt_candidates = [
    Path("correction_gpt_dpo.pt"),
    Path("week2") / "correction_gpt_dpo.pt",
    Path("correction_gpt_sft.pt"),
    Path("week2") / "correction_gpt_sft.pt",
]
_ckpt = next((p for p in _ckpt_candidates if p.is_file()), None)
if _ckpt is None:
    raise FileNotFoundError("Need correction_gpt_dpo.pt or correction_gpt_sft.pt in week2/")

model.load_state_dict(torch.load(_ckpt, map_location="cpu", weights_only=True))
model.eval()  # inference mode: no dropout training behavior
print(f"loaded weights: {_ckpt}")
print(f"tokenizer: {_tok_path} | vocab={vocab_size} | block_size={block_size}")

# ---------------------------------------------------------------------------
# Baseline: time one short generation
# ---------------------------------------------------------------------------
# prompt → encode to ids → generate 15 new tokens → decode
# We only print NEW tokens (not the whole prompt again).
#
prompt = "Ground truth: Price is $500/month\nUser said: $300\nCorrection:"
prompt_ids = tokenizer.encode(prompt)
input_ids = torch.tensor([prompt_ids], dtype=torch.long)

start = time.time()
with torch.no_grad():  # no gradients → faster / less memory at inference
    out = model.generate(input_ids, max_new_tokens=15, temperature=0.7)
end = time.time()

new_ids = out[0, len(prompt_ids) :].tolist()
print(f"Output (new tokens): {tokenizer.decode(new_ids)}")
print(f"Baseline latency: {(end - start) * 1000:.2f} ms")

n_params = sum(p.numel() for p in model.parameters())
bytes_f32 = n_params * 4  # float32 = 4 bytes per number
print(f"Model size (params): {n_params:,}")
print(f"Model size (MB): {bytes_f32 / 1e6:.2f} MB (float32)")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# loaded weights: .../correction_gpt_dpo.pt
#   Using the preference-tuned checkpoint when available.
#
# tokenizer ... vocab=260 | block_size=64
#   MUST match the checkpoint (embed is 260×64, positions length 64).
#
# Output (new tokens): ...
#   Toy model text may still look rough — this cell cares about SPEED/SIZE,
#   not perfect English. Later cells shrink/speed this same generate() call.
#
# Baseline latency: e.g. 5–50 ms (CPU varies a lot)
#   Your "before" number. Quantization should aim lower or similar with
#   smaller memory.
#
# Model size (params): ~137k
# Model size (MB): ~0.55 MB (float32)
#   Sticky: MB ≈ params × 4 / 1e6 for float32.
#   Int8 quantization idea later: ~params × 1 byte → roughly 4× smaller.


Loaded tokenizer ← correction_gpt_sft_tokenizer.json (vocab=260)
loaded weights: correction_gpt_dpo.pt
tokenizer: correction_gpt_sft_tokenizer.json | vocab=260 | block_size=64
Output (new tokens): . rthe 24 . # quiet is tes acoacaan desct
Baseline latency: 11.06 ms
Model size (params): 137,604
Model size (MB): 0.55 MB (float32)
